In [12]:
# m4.2 resampling with .resample()
# .resample() is similar to .groupby()
# .resample() is primarily used with time-series data (eg, dt.datetime)
# it groups rows by time-period and then aggregates the results
# .resample() requires a datetime index while .groupby() does not

# direction
# .resample() can "Downsample" going from Day -> Week -> Month and then aggregating
# .resample() can "Upsample" going from Month --> Week --> Day

# create a new daily demand dataset + import libraries
import pandas as pd
import numpy as np

# set a random seed so the same random data is used in all output
np.random.seed(42)

# create a date range
# .date_range() returns a fixed frequency datetime index
# you need a datetime index to use .resample()
# index will increase by 1 day starting 2024-01-01 and ending 2024-03-31
dates = pd.date_range(start="2024-01-01", end="2024-03-31", freq="D")

# create the dataframe
df = pd.DataFrame({
    "date": dates,
    "sku_id": "SKU-001",
    "daily_demand": np.random.randint(low=50, high=150, size=len(dates))
})

# set the index to "date" to ensure a datetime index for .resample()
df = df.set_index("date")

# print the last 10 rows
print(df.tail(10))
print(f"Total rows: {len(df)}")

             sku_id  daily_demand
date                             
2024-03-22  SKU-001            93
2024-03-23  SKU-001            83
2024-03-24  SKU-001           123
2024-03-25  SKU-001           111
2024-03-26  SKU-001           149
2024-03-27  SKU-001            63
2024-03-28  SKU-001           144
2024-03-29  SKU-001            97
2024-03-30  SKU-001            64
2024-03-31  SKU-001           121
Total rows: 91


In [ ]:
# exercise 2 - downsampling

# Write three resamples:

# Weekly total demand — freq="W"
df_w = df.resample("W")["daily_demand"].sum()
print(df_w)

# monthly total demand - freq = "ME"
# note: the output results in a series not a dataframe
df_m = df.resample("ME")["daily_demand"].sum()
print(df_m)

# Monthly mean, min, and max demand using .agg()
df_m_agg = df.resample("ME")["daily_demand"].agg(["mean", "min", "max"])
print(df_m_agg)

date
2024-01-07    740
2024-01-14    795
2024-01-21    578
2024-01-28    677
2024-02-04    825
2024-02-11    722
2024-02-18    617
2024-02-25    576
2024-03-03    795
2024-03-10    672
2024-03-17    517
2024-03-24    759
2024-03-31    749
Freq: W-SUN, Name: daily_demand, dtype: int64
date
2024-01-31    3166
2024-02-29    2789
2024-03-31    3067
Freq: ME, Name: daily_demand, dtype: int64
                  mean  min  max
date                            
2024-01-31  102.129032   51  149
2024-02-29   96.172414   51  141
2024-03-31   98.935484   51  149


In [ ]:
# exercise 3
# filter the resampled data

# Filter df_m for months where total demand exceeded 3000
# directly filters a series so no column df_m["daily_demand"] reference
df_m[df_m > 3000]

# Filter df_m_agg for months where average daily demand was above 100
# filters a dataframe so df_m_agg["mean"] references the "mean" column
df_m_agg[df_m_agg["mean"] > 100]

,mean,min,max
date,,,
2024-01-31,102.129032,51,149


In [ ]:
# exercise 4 - Upsampling -- increasing frequency (fewer rows --> more rows)

# upsample monthly totals back to weekly using forward fill
# propagates the last valid observation forward --> up from monthly to weekly
df_m_to_w = df_m.resample("W").ffill()
print(df_m_to_w)

# why should upsampled data be used carefully in SC forecasting?
# .bfill() and .ffill() are both used to make assumptions -- filling in gaps -- where actual observations are unavailable
# the best i use case is that you have a gap in your data
# then you become aware of a future value days, weeks, or months ahead of that data
# and you use that future value to backfill those empty values based on the actual and not a random number, guess, average or etc.
# sampled data - up or down -- should be clearly labeled -- as they can create the impression of precision when in fact you gave some number of rows the same value

date
2024-02-04    3166
2024-02-11    3166
2024-02-18    3166
2024-02-25    3166
2024-03-03    2789
2024-03-10    2789
2024-03-17    2789
2024-03-24    2789
2024-03-31    3067
Freq: W-SUN, Name: daily_demand, dtype: int64
